# Cómo usar notebooks en este proyecto

Este notebook no es de ML — es una guía de uso del entorno. Explica **cómo interactuar** con los notebooks del proyecto a través de casos de uso concretos.

**Antes de empezar:** asegúrate de que el contenedor está levantado y el kernel está seleccionado.
- Esquina superior derecha → `Select Kernel` → `Python (videocatalog)`
- Si no aparece ningún kernel: `docker compose up -d` en el terminal, luego recargar la ventana (`Ctrl+Shift+P` → `Reload Window`)

---
## Sección 1 — Anatomía de un notebook

Un notebook está compuesto de **celdas**. Hay dos tipos:

- **Celda de código** (fondo gris claro): contiene Python. Se ejecuta con `Shift+Enter`.
- **Celda Markdown** (fondo blanco): contiene texto formateado. También se renderiza con `Shift+Enter`.

**El estado del kernel es compartido entre celdas.** Si defines una variable en la celda 3, puedes usarla en la celda 7. Si reinicias el kernel (`Kernel → Restart`), todas las variables se pierden y hay que re-ejecutar desde el principio.

**El orden de ejecución importa.** Los números entre corchetes a la izquierda de cada celda (`[1]`, `[2]`, ...) indican el orden en que se ejecutaron. Si ves `[*]`, la celda está ejecutándose. Si ves celdas ejecutadas en orden no secuencial, puede haber dependencias rotas.

**Flujo habitual de trabajo:**
1. Abrir el notebook
2. `Kernel → Restart & Run All` para ejecutar todo desde cero
3. Modificar un parámetro en alguna celda
4. `Shift+Enter` para re-ejecutar esa celda y ver el resultado
5. Continuar modificando y re-ejecutando

Ejecuta la celda siguiente para verificar que el kernel funciona:

In [ ]:
# Celda de verificación — ejecuta con Shift+Enter
import sys
print(f"Python: {sys.version}")
print(f"Directorio de trabajo: {__import__('os').getcwd()}")

# Verificar que las librerías del proyecto están instaladas
import pandas, sklearn, joblib, matplotlib
print(f"pandas: {pandas.__version__}")
print(f"scikit-learn: {sklearn.__version__}")
print("\n✓ Todo correcto — puedes continuar")

---
## Sección 2 — Caso de uso 1: Explorar `labels.csv`

El caso más frecuente: cargar el dataset y responder preguntas sobre él.

**Patrón:** cargar → inspeccionar → visualizar → modificar un parámetro → re-ejecutar.

**Prerequisito:** haber ejecutado `python -m src.label --export` en el terminal para generar `data/labels.csv`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

CSV_PATH = Path("../data/labels.csv")

if not CSV_PATH.exists():
    print("⚠ data/labels.csv no existe todavía.")
    print("  Ejecuta en el terminal:")
    print("  docker compose exec videocatalog python -m src.label --export")
else:
    df = pd.read_csv(CSV_PATH)
    print(f"Dataset cargado: {len(df)} vídeos, {len(df.columns)} columnas")
    print(f"Columnas: {df.columns.tolist()}")
    df.head()

In [ ]:
# Distribución de categorías
# PRUEBA: cambia el valor de TOP_N y re-ejecuta con Shift+Enter
TOP_N = 10  # ← modifica este número

counts = df["label"].value_counts().head(TOP_N)

fig, ax = plt.subplots(figsize=(10, 4))
counts.plot(kind="bar", ax=ax, color="steelblue", alpha=0.8)
ax.set_title(f"Top {TOP_N} categorías por número de vídeos")
ax.set_ylabel("Número de vídeos")
ax.set_xlabel("Categoría")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

print(f"\nFactor de desbalance: {counts.iloc[0]} / {counts.iloc[-1]} = {counts.iloc[0]/counts.iloc[-1]:.1f}x")

In [ ]:
# Distribución de rule_score
# PRUEBA: cambia THRESHOLD y observa cómo cambia el porcentaje de vídeos afectados
THRESHOLD = 0.2  # ← modifica este valor (prueba 0.1, 0.3, 0.5)

fig, ax = plt.subplots(figsize=(10, 4))
df["rule_score"].hist(bins=25, ax=ax, color="steelblue", alpha=0.8)
ax.axvline(THRESHOLD, color="red", linestyle="--", linewidth=2, label=f"threshold = {THRESHOLD}")
ax.set_title("Distribución de rule_score (confianza del clasificador de reglas)")
ax.set_xlabel("Score")
ax.set_ylabel("Número de vídeos")
ax.legend()
plt.tight_layout()
plt.show()

bajo = (df["rule_score"] < THRESHOLD).sum()
print(f"Vídeos con score < {THRESHOLD}: {bajo} ({100*bajo/len(df):.1f}%)")
print(f"→ Estos son los candidatos prioritarios para etiquetar manualmente (Task 9)")

**Lo que acabas de ver:** el patrón de trabajo en los notebooks de exploración (Task 10).
Cargas datos, produces un gráfico, modificas un parámetro (`TOP_N`, `THRESHOLD`),
re-ejecutas la celda y el gráfico se actualiza instantáneamente.

No necesitas re-ejecutar el notebook completo para ver el efecto de un cambio —
solo la celda que modificaste (y las que dependan de ella).

---
## Sección 3 — Caso de uso 2: Usar el modelo entrenado

Cargar `model.joblib` y predecir sobre títulos que tú escribes.

**Prerequisito:** haber completado Task 12 (el notebook de entrenamiento debe haber generado `data/model.joblib`).

In [ ]:
import joblib
import json
from pathlib import Path

MODEL_PATH   = Path("../data/model.joblib")
METRICS_PATH = Path("../data/model_metrics.json")

if not MODEL_PATH.exists():
    print("⚠ data/model.joblib no existe todavía.")
    print("  Completa Task 12 (notebooks/03_train_classifier.ipynb) primero.")
else:
    pipeline = joblib.load(MODEL_PATH)
    print(f"✓ Modelo cargado desde {MODEL_PATH}")
    print(f"  Tipo: {type(pipeline)}")
    print(f"  Categorías: {list(pipeline.classes_)}")

    if METRICS_PATH.exists():
        with open(METRICS_PATH) as f:
            metrics = json.load(f)
        print(f"\n  F1 macro: {metrics['f1_macro']:.3f}")
        print(f"  Entrenado con: {metrics['n_train']} ejemplos")
        print(f"  Fecha: {metrics['trained_at'][:10]}")

In [ ]:
# PRUEBA: modifica esta lista y re-ejecuta para ver qué predice el modelo
TITULOS_DE_PRUEBA = [
    "curso completo python desde cero",
    "docker networking introduccion contenedores",
    "introduccion al machine learning scikit learn",
    "javascript react hooks tutorial",
    "linux bash scripting automatizacion",
    "ciberseguridad hacking etico pentest",
    "arduino raspberry pi iot proyecto",
    # Añade tus propios títulos aquí ↓
]

predicciones  = pipeline.predict(TITULOS_DE_PRUEBA)
probabilidades = pipeline.predict_proba(TITULOS_DE_PRUEBA)

print(f"{'Título':<50} {'Categoría':<15} {'Confianza':>10}")
print("-" * 78)
for titulo, cat, proba in zip(TITULOS_DE_PRUEBA, predicciones, probabilidades):
    confianza = proba.max()
    barra = "█" * int(confianza * 20)
    titulo_corto = titulo[:48] + ".." if len(titulo) > 48 else titulo
    print(f"{titulo_corto:<50} {cat:<15} {confianza:>8.2f}  {barra}")

In [ ]:
# Ver las probabilidades completas para un título concreto
# PRUEBA: cambia el índice (0, 1, 2...) para inspeccionar distintos títulos
INDICE = 0  # ← índice del título en TITULOS_DE_PRUEBA

titulo = TITULOS_DE_PRUEBA[INDICE]
proba_fila = probabilidades[INDICE]

# Ordenar categorías por probabilidad descendente
orden = proba_fila.argsort()[::-1]

print(f"Título: '{titulo}'")
print(f"Predicción: {pipeline.classes_[orden[0]]} (confianza: {proba_fila[orden[0]]:.3f})")
print(f"\nDistribución de probabilidades:")
for i in orden:
    cat = pipeline.classes_[i]
    p = proba_fila[i]
    if p > 0.005:  # solo mostrar categorías con probabilidad > 0.5%
        barra = "█" * int(p * 40)
        print(f"  {cat:<20} {p:>6.3f}  {barra}")

**Lo que acabas de ver:** cómo cargar un modelo serializado con `joblib.load` y
obtener tanto la categoría predicha (`predict`) como la distribución de probabilidades
sobre todas las categorías (`predict_proba`).

La confianza es la probabilidad máxima — cuánto de seguro está el modelo.
Un título con confianza 0.95 en Python está casi seguro. Uno con confianza 0.35
en Python y 0.30 en IA_ML es un caso ambiguo que podría beneficiarse de revisión manual.

En Task 13, `src/classifier.py` hace exactamente esto: carga el modelo, llama a
`predict_proba`, y devuelve la categoría con la confianza como un dict.

---
## Sección 4 — Caso de uso 3: Comparar configuraciones

Experimentar con hiperparámetros: cambiar un parámetro, reentrenar, comparar F1.

**Este es el flujo de experimentación ML.** No reemplaza al ciclo de reentrenamiento
de Task 14 — es un sandbox para explorar qué configuración funciona mejor antes
de fijarla en el código de producción.

**Prerequisito:** `data/labels.csv` con suficientes ejemplos etiquetados (al menos 50 por categoría para que los resultados sean significativos).

In [ ]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.metrics import f1_score
import numpy as np

CSV_PATH = Path("../data/labels.csv")
if not CSV_PATH.exists():
    print("⚠ Necesitas data/labels.csv para esta sección.")
else:
    df = pd.read_csv(CSV_PATH)
    X = df["clean_title"].fillna("").tolist()
    y = df["label"].tolist()
    print(f"Dataset: {len(X)} ejemplos, {len(set(y))} categorías")

In [ ]:
# Comparar ngram_range=(1,1) vs (1,2) vs (1,3)
# PRUEBA: añade o quita configuraciones de la lista y re-ejecuta
# Cada configuración es (nombre, ngram_range, max_features)

CONFIGURACIONES = [
    ("Unigramas (1,1)",       (1, 1), 3000),
    ("Bigramas  (1,2)",       (1, 2), 5000),   # ← el que usa el proyecto por defecto
    ("Trigramas (1,3)",       (1, 3), 8000),
    ("Bigramas pequeño",      (1, 2), 1000),   # ← menos features, más rápido
]

resultados = []

for nombre, ngram, max_f in CONFIGURACIONES:
    pipeline_exp = Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=ngram, max_features=max_f)),
        ("clf",   LogisticRegression(class_weight="balanced", max_iter=500, random_state=42)),
    ])
    # cross_val_score entrena y evalúa 5 veces con splits distintos (más robusto que un split único)
    scores = cross_val_score(pipeline_exp, X, y, cv=5, scoring="f1_macro", n_jobs=-1)
    resultados.append({
        "Configuración": nombre,
        "F1 macro (media)": round(scores.mean(), 3),
        "F1 macro (±std)": round(scores.std(), 3),
    })
    print(f"  {nombre:<25} F1={scores.mean():.3f} ± {scores.std():.3f}")

print("\n✓ Comparación completa")

In [ ]:
# Visualizar la comparación
import matplotlib.pyplot as plt
import numpy as np

nombres = [r["Configuración"] for r in resultados]
medias  = [r["F1 macro (media)"] for r in resultados]
stds    = [r["F1 macro (±std)"] for r in resultados]

x = np.arange(len(nombres))
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(x, medias, yerr=stds, capsize=5, color="steelblue", alpha=0.8)

# Resaltar la mejor configuración
mejor_idx = np.argmax(medias)
bars[mejor_idx].set_color("orange")

ax.set_xticks(x)
ax.set_xticklabels(nombres, rotation=15, ha="right")
ax.set_ylabel("F1 macro (CV 5-fold)")
ax.set_title("Comparación de configuraciones TF-IDF")
ax.set_ylim(0, 1.05)

# Anotar los valores
for bar, media in zip(bars, medias):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{media:.3f}", ha="center", va="bottom", fontsize=10)

plt.tight_layout()
plt.show()

print(f"\nMejor configuración: {nombres[mejor_idx]} (F1={medias[mejor_idx]:.3f})")

**Lo que acabas de ver:** validación cruzada (`cross_val_score` con `cv=5`) en lugar
de un único train/test split. Con 5 folds, el dataset se divide 5 veces de formas
distintas y el F1 se promedia — da una estimación más robusta del rendimiento real
que un único split aleatorio.

La barra de error (±std) indica cuánto varía el rendimiento entre los 5 folds.
Si la varianza es alta, el modelo es inestable (probablemente por poco datos).
Si es baja, el resultado es reproducible.

**Cómo usar esto en la práctica:** cuando en Task 14 quieras probar un cambio
(más features, un parámetro distinto, filtrar labels por `label_source`), haz
el experimento aquí primero. Si el F1 sube, ajusta `src/train.py` con ese cambio
y ejecuta el reentrenamiento formal.

---
## Sección 5 — Exportar resultados

Cómo guardar gráficos como PNG y tablas como CSV desde el notebook.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

# Directorio de salida
OUTPUT_DIR = Path("../data/exports")
OUTPUT_DIR.mkdir(exist_ok=True)

# --- Guardar un gráfico como PNG ---

# Recrear el gráfico de distribución de categorías (si df existe)
try:
    fig, ax = plt.subplots(figsize=(12, 5))
    df["label"].value_counts().plot(kind="bar", ax=ax, color="steelblue", alpha=0.8)
    ax.set_title("Distribución de categorías")
    ax.set_ylabel("Número de vídeos")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()

    # Guardar — dpi=150 para buena resolución sin fichero enorme
    png_path = OUTPUT_DIR / "distribucion_categorias.png"
    fig.savefig(png_path, dpi=150, bbox_inches="tight")
    plt.close(fig)  # cerrar para no mostrar dos veces
    print(f"✓ Gráfico guardado: {png_path}")
except NameError:
    print("⚠ Ejecuta la Sección 2 primero para cargar df")

# --- Guardar una tabla como CSV ---
try:
    # Tabla de resultados de la comparación de configuraciones (Sección 4)
    resultados_df = pd.DataFrame(resultados)
    csv_path = OUTPUT_DIR / "comparacion_configuraciones.csv"
    resultados_df.to_csv(csv_path, index=False)
    print(f"✓ Tabla guardada: {csv_path}")
    print(resultados_df.to_string(index=False))
except NameError:
    print("⚠ Ejecuta la Sección 4 primero para generar resultados")

In [ ]:
# Verificar que los ficheros se guardaron correctamente
print("Ficheros en data/exports/:")
for f in sorted(OUTPUT_DIR.iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:<45} {size_kb:>6.1f} KB")

**Resumen de lo que has visto en este notebook:**

| Caso de uso | Patrón clave |
|---|---|
| Explorar datos | Cargar CSV → filtrar → graficar → modificar parámetro → re-ejecutar |
| Usar el modelo | `joblib.load` → `predict` / `predict_proba` → inspeccionar confianza |
| Comparar configuraciones | `cross_val_score` → gráfico comparativo → elegir la mejor |
| Exportar resultados | `fig.savefig` para PNG, `df.to_csv` para CSV |

**Los notebooks del proyecto** (`01_explore_data.ipynb` al `05_rules_vs_model.ipynb`)
son variantes más elaboradas de estos mismos patrones aplicados a los objetivos de ML.

Para cualquier duda sobre una llamada específica (`pipeline.predict_proba`,
`cross_val_score`, `f1_score`, etc.), consulta `docs/python-para-ml.md`.